# PKU6 clinical fMRI time-series and regional-metric extraction

This notebook processes the clinical data . It matches the original workflow by loading full motion parameters and basic WM/CSF signals from fMRIPrep.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    PROJECT_ROOT = (PROJECT_ROOT / "..").resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    raise FileNotFoundError("Run this notebook from the RISE repository root or pipeline directory")
from rise import (
    build_fmriprep_run_inputs,
    extract_dataset,
    load_config,
    prepare_atlases,
)

## Configuration

In [ ]:
CONFIG_FILE = PROJECT_ROOT / "configs" / "pku6_timeseries_parameters.jsonc"
config = load_config(CONFIG_FILE)
config


## Discover PKU6 fMRIPrep runs

By default, all `sub-*` directories under the configured fMRIPrep derivatives root are discovered. The saved `pku6_ids.csv` remains available when a defined subset is needed.

In [ ]:
run_inputs = build_fmriprep_run_inputs(config=config)
subjects = sorted({item["id"] for item in run_inputs})
print(f"Found {len(run_inputs)} fMRIPrep runs for {len(subjects)} subjects.")
run_inputs[:2]

In [ ]:
missing_confounds = [item for item in run_inputs if "confounds_file" not in item]
print(f"Runs without a matching confounds TSV: {len(missing_confounds)}")
missing_confounds[:10]

## Prepare atlas files

The atlases and the brain mask are resampled once to the PKU6 fMRIPrep BOLD grid and cached under `data/clinical/PKU6/atlas_cache`.

In [ ]:
if not run_inputs:
    raise RuntimeError("No PKU6 runs were found. Check dataset_root and pku6_ids.csv.")
atlas_cache = prepare_atlases(run_inputs[0]["nii_file"], config, force=False)
print(f"Prepared atlases: {atlas_cache}")

## Run extraction

Outputs are written under `data/clinical/PKU6/processed_data/indiv/<subject>/func/ses-1_task-rest_run-1/data`.

In [ ]:
results = extract_dataset(
    run_inputs[:2],
    config,
    n_processes=config["n_processes"],
)
results[:2]